# Artwork 3

In [1]:
# first we need to import our visualization libraries and any other related components

import json # need to parse geojson
import re # needed for wordcloud example


import pandas as pd # using data frames with our plotly visualization
import plotly.express as px # px shorthand is used in plotly examples


from IPython.display import HTML, IFrame # needed to display pydeck visualization in the notebook


In [60]:
import requests
import time

base_url = "https://collectionapi.metmuseum.org"
search_path = "/public/collection/v1.1/search"

search_query = "Wapisiana"
limit = 500
offset = 0

all_results = []

while True:
    url = (
        f"{base_url}{search_path}"
        f"?q={search_query.replace(' ', '%20')}"
        f"&artistOrCulture=true"
        f"&limit={limit}"
        f"&offset={offset}"
    )

    response = requests.get(url)
    response.raise_for_status()

    data = response.json()

    # Get the object IDs returned by this page
    object_ids = data.get("objectIDs") or []

    # Stop if there are no more results
    if not object_ids:
        break

    all_results.extend(object_ids)

    print(f"Collected {len(all_results)} records so far")

    # Move to the next page
    offset += limit

    time.sleep(1)

print(f"Total records collected: {len(all_results)}")

print(url)

Collected 5 records so far
Total records collected: 5
https://collectionapi.metmuseum.org/public/collection/v1.1/search?q=Wapisiana&artistOrCulture=true&limit=500&offset=500


In [61]:
complete_Wapisiana_objects = all_results

print(len(complete_Wapisiana_objects))
print(complete_Wapisiana_objects)

5
[502976, 502977, 502978, 504970, 501285]


In [62]:
from time import sleep 

object_path = "/public/collection/v1/objects/" 
all_Wapisiana_details = []

for Wapisiana_artwork in complete_Wapisiana_objects:
    url = f"{base_url}{object_path}{Wapisiana_artwork}"
    response_object = requests.get(url) 
    
    # Check if the API request was successful (status code 200)
    if response_object.ok:
        all_Wapisiana_details.append(response_object.json())
        print(f"grabbed details from objectID {Wapisiana_artwork} with status code {response_object.status_code}")
    else:
        # If it's a 404 or any other error, skip it and don't append it to the list
        print(f"⚠️ SKIPPED objectID {Wapisiana_artwork} - Received status code {response_object.status_code}")
        
    sleep(1) 

print(f"Total valid objects captured: {len(all_Wapisiana_details)}") 
if all_Wapisiana_details:
    print(all_Wapisiana_details[0])

grabbed details from objectID 502976 with status code 200
grabbed details from objectID 502977 with status code 200
grabbed details from objectID 502978 with status code 200
grabbed details from objectID 504970 with status code 200
grabbed details from objectID 501285 with status code 200
Total valid objects captured: 5
{'objectID': 502976, 'isHighlight': False, 'accessionNumber': '08.54.2', 'accessionYear': '1908', 'isPublicDomain': False, 'primaryImage': '', 'primaryImageSmall': '', 'additionalImages': [], 'constituents': None, 'department': 'Musical Instruments', 'objectName': 'Wat-Sa-Pua (bone flute)', 'title': 'Wat-Sa-Pua (bone flute)', 'culture': 'Native American (Wapisiana, probably)', 'period': '', 'dynasty': '', 'reign': '', 'portfolio': '', 'artistRole': '', 'artistPrefix': '', 'artistDisplayName': '', 'artistDisplayBio': '', 'artistSuffix': '', 'artistAlphaSort': '', 'artistNationality': '', 'artistBeginDate': '', 'artistEndDate': '', 'artistGender': '', 'artistWikidata_URL'

In [63]:
import csv   # brings in all the csv-related functions

with open("all_Wapisiana_artworks.csv", "w", encoding="utf-8") as file: # initiate the opening of a file, and write a new file
    writer = csv.DictWriter(file, all_Wapisiana_details[0].keys()) # this creates the columns in the CSV file from the keys
    writer.writeheader() # Creates the first row of the CSV, containing the HEaders
    writer.writerows(all_Wapisiana_details) # goes through every item in the yoko_ono_artworks list and put it in the CSV file
    # you should see it in your folder after running.

In [71]:
import pandas as pd

# List your CSV files in the order you want to combine them
csv_files = [
    "all_nativeamerican_artworks.csv",
    "all_apache_artworks.csv",
    "all_cahuilla_artworks.csv",
    "all_hopi_artworks.csv",
    "all_sioux_artworks.csv",
    "all_shawnee_artworks.csv",
    "all_xaadas_haida_artworks.csv",
    "all_mission_artworks.csv",
    "all_seneca_artworks.csv",
    "all_zuni_artworks.csv",
    "all_arapaho_artworks.csv",
    "all_Navaho_artworks.csv",
    "all_Iroquois_artworks.csv",
    "all_Inuit_artworks.csv",
    "all_Pueblo_artworks.csv",
    "all_Pima_artworks.csv",
    "all_Shoshone_artworks.csv",
    "all_Nootka_artworks.csv",
    "all_Hupa_artworks.csv",
    
]

# Read and combine all CSV files
dataframes = [pd.read_csv(file) for file in csv_files]

# Combine all datasets
combined = pd.concat(dataframes, ignore_index=True)

# Clean objectID values
combined["objectID"] = (
    combined["objectID"]
    .astype("string")
    .str.strip()
)

# Remove rows with missing objectIDs
combined = combined[combined["objectID"].notna() & combined["objectID"].ne("")]

# Remove duplicate objectIDs, keeping the first occurrence
combined = combined.drop_duplicates(
    subset="objectID",
    keep="first"
)

# Save combined dataset
combined.to_csv("artworks_combined.csv", index=False)

# Display results
print("Total CSV files:", len(csv_files))
print("Combined unique records:", len(combined))
print("Final dataset saved as artworks_combined.csv")

Total CSV files: 18
Combined unique records: 898
Final dataset saved as artworks_combined.csv 0


In [72]:
# Read CSV files
df1 = pd.read_csv("all_nativeamerican_artworks.csv")
df2 = pd.read_csv("artworks_combined.csv")

# Count rows
print("CSV 1 rows:", len(df1))
print("CSV 2 rows:", len(df2))

# Compare
print("Difference:", len(df1) - len(df2))

CSV 1 rows: 501
CSV 2 rows: 898
Difference: -397


In [162]:
# round 2!

# List your CSV files in the order you want to combine them
csv_files = [
    "artworks_combined.csv",
    "all_Kiowa_artworks.csv",
    "all_Yupik_artworks.csv",
    "all_Pomo_artworks.csv",
    "all_Haida_artworks.csv",
    "all_Mikmaq_artworks.csv",
    "all_Wasco_artworks.csv",
    "all_Acoma_artworks.csv",
    "all_Tewa_artworks.csv",
    "all_Washoe_artworks.csv",
    "all_Seminole_artworks.csv",
    "all_Comanche_artworks.csv",
    "all_Crow_artworks.csv",
    "all_Tsimshian_artworks.csv",
    "all_Chippewa_artworks.csv",
    "all_Ojibwa_artworks.csv",
    "all_Alutiiq_artworks.csv",
    "all_Navajo_artworks.csv"
    
]

# Read and combine all CSV files
dataframes = [pd.read_csv(file) for file in csv_files]

# Combine all datasets
combined = pd.concat(dataframes, ignore_index=True)

# Clean objectID values
combined["objectID"] = (
    combined["objectID"]
    .astype("string")
    .str.strip()
)

# Remove rows with missing objectIDs
combined = combined[combined["objectID"].notna() & combined["objectID"].ne("")]

# Remove duplicate objectIDs, keeping the first occurrence
combined = combined.drop_duplicates(
    subset="objectID",
    keep="first"
)

# Save combined dataset
combined.to_csv("artworks_combined_2.csv", index=False)

# Display results
print("Total CSV files:", len(csv_files))
print("Combined unique records:", len(combined))
print("Final dataset saved as artworks_combined_2.csv")

Total CSV files: 18
Combined unique records: 990
Final dataset saved as artworks_combined_2.csv


In [163]:
# Read CSV files
df1 = pd.read_csv("artworks_combined.csv")
df2 = pd.read_csv("artworks_combined_2.csv")

# Count rows
print("CSV 1 rows:", len(df1))
print("CSV 2 rows:", len(df2))

# Compare
print("Difference:", len(df1) - len(df2))

CSV 1 rows: 898
CSV 2 rows: 990
Difference: -92


In [64]:
# Round 3!

# List your CSV files in the order you want to combine them
csv_files = [
    "artworks_combined_2.csv",
    "all_Osage_artworks.csv",
    "all_Kwakwakawakw_artworks.csv",
    "all_Penobscot_artworks.csv",
    "all_Cheyenne_artworks.csv",
    "all_Makah_artworks.csv",
    "all_Miwok_artworks.csv",
    "all_Kwakuitl_artworks.csv",
    "all_Naskapi_artworks.csv",
    "all_Yakutat_artworks.csv",
    "all_Wapisiana_artworks.csv"
]

# Read and combine all CSV files
dataframes = [pd.read_csv(file) for file in csv_files]

# Combine all datasets
combined = pd.concat(dataframes, ignore_index=True)

# Clean objectID values
combined["objectID"] = (
    combined["objectID"]
    .astype("string")
    .str.strip()
)

# Remove rows with missing objectIDs
combined = combined[combined["objectID"].notna() & combined["objectID"].ne("")]

# Remove duplicate objectIDs, keeping the first occurrence
combined = combined.drop_duplicates(
    subset="objectID",
    keep="first"
)

# Save combined dataset
combined.to_csv("artworks_combined_3.csv", index=False)

# Display results
print("Total CSV files:", len(csv_files))
print("Combined unique records:", len(combined))
print("Final dataset saved as artworks_combined_3.csv")

Total CSV files: 11
Combined unique records: 1120
Final dataset saved as artworks_combined_3.csv


In [65]:
# comparison

# Read CSV files
df1 = pd.read_csv("artworks_combined_2.csv")
df2 = pd.read_csv("artworks_combined_3.csv")

# Count rows
print("CSV 1 rows:", len(df1))
print("CSV 2 rows:", len(df2))

# Compare
print("Difference:", len(df1) - len(df2))

CSV 1 rows: 990
CSV 2 rows: 1120
Difference: -130


In [70]:
# Count original records
original_count = len(combined)

# Keep only entries WITH a state value
nativeamerican_with_state = combined[
    combined["state"].notna() &
    combined["state"].astype("string").str.strip().ne("")
].copy()

# Save as a new CSV
nativeamerican_with_state.to_csv(
    "artworks_combined_3_with_state.csv",
    index=False
)

# Display results
print("Original records:", original_count)
print("Records WITH state:", len(nativeamerican_with_state))
print("Records removed:", original_count - len(nativeamerican_with_state))
print("New CSV saved: artworks_combined_3_with_state.csv")

Original records: 1120
Records WITH state: 897
Records removed: 223
New CSV saved: artworks_combined_3_with_state.csv


In [71]:
# removing unwanted characters
# Count original records
original_count = len(combined)

# Keep only entries WITH a state value
nativeamerican_with_state = combined[
    combined["state"].notna() &
    combined["state"].astype("string").str.strip().ne("")
].copy()

# Remove entries containing unwanted characters or words in state
unwanted_pattern = r"\band\b|\bor\b|/|,|\?|\&"

nativeamerican_with_state = nativeamerican_with_state[
    ~nativeamerican_with_state["state"].astype("string").str.contains(
        unwanted_pattern,
        case=False,
        regex=True,
        na=False
    )
].copy()

# Save as a new CSV
nativeamerican_with_state.to_csv(
    "artworks_combined_3_with_state_cleaned.csv",
    index=False
)

# Display results
print("Original records:", original_count)
print("Records with state after cleaning:", len(nativeamerican_with_state))
print("Records removed:", original_count - len(nativeamerican_with_state))
print("New CSV saved: artworks_combined_3_with_state_cleaned.csv")

Original records: 1120
Records with state after cleaning: 783
Records removed: 337
New CSV saved: artworks_combined_3_with_state_cleaned.csv
